In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
from tqdm.auto import tqdm
from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_merton_inputs, prepare_nig_inputs
)
from pd_estim_A.data.cds_df import get_cds_panel
from pd_estim_A.models.nig.nig_apath import (
    NIGParams,
    invert_assets_weekly_for_firm,
)

from pd_estim_A.models.nig.nig_em import (
    fit_nig_params_from_weekly_assets,
)

from pd_estim_A.models.nig.nig_pd import (
    pd_terminal_nig_weekly,
    pd_weekly_one_firm,
)

c:\Users\vkeenan\AppData\Local\miniconda3\envs\Accenture\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# Load data and prepare panels
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()


nig_df, em_cache = prepare_nig_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt, build_em=False)
print(nig_df.head())
print(nig_df.shape)
print(nig_df.describe())

c:\Users\vkeenan\OneDrive - Delft University of Technology\Documents\University\QRM\Accenture Project\code\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\vkeenan\OneDrive - Delft University of Technology\Documents\University\QRM\Accenture Project\code\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10
    gvkey       date             E          isin  \
0  100022 2012-01-03  3.328431e+10  DE0005190003   
1  100080 2012-01-03  4.268705e+10  DE000BAY0017   
2  100312 2012-01-03  1.469717e+09  DE0007030009   
3  100581 2012-01-03  4.935351e+10  FR0000120321   
4  100957 2012-01-03  2.931

In [ ]:
# call cds panel and merge
cds = get_cds_panel(
    project_root= Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)
# ensure types
merton = nig_df.copy()
merton["gvkey"] = merton["gvkey"].astype(str)
merton["date"]  = pd.to_datetime(merton["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"]  = pd.to_datetime(cds["date"])

# keep only firms that exist in BOTH (drop firms with no CDS)
common_gv = sorted(set(merton["gvkey"].unique()) & set(cds["gvkey"].unique()))
merton = merton[merton["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# also drop CDS rows whose dates are outside merged's date range
dmin, dmax = merton["date"].min(), merton["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge-asof onto merged's dates (direction='backward')
merton = merton.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds    = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    merton,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

# drop rows where CDS still missing
nig_df = merged_cds.dropna(subset=["cds"]).reset_index(drop=True)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())

In [ ]:
# Cell 2 — rolling configuration

TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_ENDING = "W-FRI"

T_INV = 1.0                  # 1Y maturity used in inversion
HORIZON_WEEKS = 52.0         # 1Y PD horizon on weekly scale

DATA_END = pd.Timestamp("2024-12-31")   # keep equal to Merton if you want clean comparison
LAST_TRAIN_END = DATA_END - pd.offsets.QuarterEnd(1)

MIN_DAILY_ROWS = 10
MIN_WEEKLY_RETURNS = 60      # require enough weekly implied asset returns in each 2Y window

P0 = NIGParams(alpha=15.0, beta=-3.0, delta=0.20, mu=0.00)

INVERT_U = 120.0
INVERT_N = 2000

EM_MAX_ITER = 80
EM_TOL = 1e-6

MAX_FIRMS = None
MAX_WINDOWS = None

In [ ]:
# Cell 3 — panel preparation
# Assumes nig_df already exists and has at least:
# gvkey, date, E, L, r
# plus optionally company, country_iso, etc.

panel = nig_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

needed_cols = ["gvkey", "date", "company", "E", "L", "r"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

for c in ["E", "L", "r"]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = (
    panel.dropna(subset=["gvkey", "date", "E", "L", "r"])
         .query("E > 0 and L > 0")
         .sort_values(["gvkey", "date"])
         .reset_index(drop=True)
)

firm_daily = {}
for gvkey, g in panel.groupby("gvkey", sort=False):
    g = g.sort_values("date").groupby("date", as_index=False).last()
    firm_daily[gvkey] = g.set_index("date")

gvkeys_all = sorted(firm_daily.keys())
if MAX_FIRMS is not None:
    gvkeys_all = gvkeys_all[:int(MAX_FIRMS)]

print("Firms loaded:", len(firm_daily), "| Firms in run:", len(gvkeys_all))
print("Panel date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("LAST_TRAIN_END:", LAST_TRAIN_END.date(), "| DATA_END:", DATA_END.date())
display(panel.head())

Firms loaded: 21 | Firms in run: 21
Panel date range: 2014-01-01 to 2025-12-19
LAST_TRAIN_END: 2024-09-30 | DATA_END: 2024-12-31


,gvkey,date,company,E,L,r
0,100022,2014-01-01,BAYERISCHE MOTOREN WERKE AKT,5.130203e+10,1.014480e+11,0.000942
1,100022,2014-01-02,BAYERISCHE MOTOREN WERKE AKT,5.029068e+10,1.014480e+11,0.001069
2,100022,2014-01-03,BAYERISCHE MOTOREN WERKE AKT,5.055556e+10,1.014480e+11,0.000991
3,100022,2014-01-06,BAYERISCHE MOTOREN WERKE AKT,4.995958e+10,1.014480e+11,0.001006
4,100022,2014-01-07,BAYERISCHE MOTOREN WERKE AKT,5.029670e+10,1.014480e+11,0.001057


In [ ]:
# Cell 4 — build the rolling window schedule

global_min_date = panel["date"].min()
earliest_end = global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1)

train_ends = pd.date_range(start=earliest_end, end=LAST_TRAIN_END, freq=STEP_FREQ)
train_ends = pd.to_datetime(train_ends)

if MAX_WINDOWS is not None:
    train_ends = train_ends[:int(MAX_WINDOWS)]

windows = []
for train_end in train_ends:
    train_start = train_end - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    oos_start = train_end + pd.Timedelta(days=1)
    oos_end = train_end + pd.offsets.QuarterEnd(1)

    windows.append(
        {
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "oos_start": pd.Timestamp(oos_start),
            "oos_end": pd.Timestamp(oos_end),
        }
    )

windows_df = pd.DataFrame(windows)
display(windows_df.head())
display(windows_df.tail())
print("n_windows:", len(windows_df))

,train_start,train_end,oos_start,oos_end
0,2014-01-01,2015-12-31,2016-01-01,2016-03-31
1,2014-04-01,2016-03-31,2016-04-01,2016-06-30
2,2014-07-01,2016-06-30,2016-07-01,2016-09-30
3,2014-10-01,2016-09-30,2016-10-01,2016-12-31
4,2015-01-01,2016-12-31,2017-01-01,2017-03-31


,train_start,train_end,oos_start,oos_end
31,2021-10-01,2023-09-30,2023-10-01,2023-12-31
32,2022-01-01,2023-12-31,2024-01-01,2024-03-31
33,2022-04-01,2024-03-31,2024-04-01,2024-06-30
34,2022-07-01,2024-06-30,2024-07-01,2024-09-30
35,2022-10-01,2024-09-30,2024-10-01,2024-12-31


n_windows: 36


In [ ]:
# Cell 5 — thin helpers

def row_to_nig_params(row: pd.Series) -> NIGParams:
    return NIGParams(
        alpha=float(row["alpha"]),
        beta=float(row["beta"]),
        delta=float(row["delta"]),
        mu=float(row["mu"]),
    )


def fit_one_nig_window(
    assets_weekly: pd.DataFrame,
    p_start: NIGParams,
    *,
    em_max_iter: int = 80,
    em_tol: float = 1e-6,
    alpha_min: float = 3.0,
) -> tuple[NIGParams, pd.DataFrame]:
    """
    One EM fit on the full current training window, using the existing rolling fitter
    in single-window mode.
    """
    n_ret = int(assets_weekly["dlogA"].notna().sum())
    if n_ret < 10:
        raise ValueError(f"Too few weekly implied asset returns: {n_ret}")

    upd = fit_nig_params_from_weekly_assets(
        assets_weekly,
        p0=p_start,
        window_weeks=n_ret,
        refit_every=n_ret,
        em_max_iter=em_max_iter,
        em_tol=em_tol,
        alpha_min=alpha_min,
        use_theta=True,
        prefer_precomputed_theta=True,
        verbose=False,
    )

    if upd.empty:
        raise RuntimeError("Single-window NIG fit produced no update row.")

    return row_to_nig_params(upd.iloc[-1]), upd


def one_row_param_update(train_end: pd.Timestamp, p_hat: NIGParams) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "date": [pd.Timestamp(train_end)],
            "alpha": [float(p_hat.alpha)],
            "beta": [float(p_hat.beta)],
            "delta": [float(p_hat.delta)],
            "mu": [float(p_hat.mu)],
        }
    )


def nig_annual_to_weekly(p: NIGParams, weeks_per_year: float = 52.0) -> NIGParams:
    return NIGParams(
        alpha=float(p.alpha),
        beta=float(p.beta),
        delta=float(p.delta) / float(weeks_per_year),
        mu=float(p.mu) / float(weeks_per_year),
    )

In [ ]:
# Cell 6 — rolling one-pass NIG estimation

roll_summary_rows = []
roll_weekly_is_rows = []
roll_weekly_oos_rows = []

t0_all = time.time()

# warm start ONLY for EM
prev_p_em = {}

# stable pricing / inversion seed
P0_INV = NIGParams(alpha=10.0, beta=0.0, delta=0.20, mu=0.00)
P0_EM = nig_annual_to_weekly(P0_INV)

for w in tqdm(windows, desc="Rolling NIG windows"):
    train_start = w["train_start"]
    train_end   = w["train_end"]
    oos_start   = w["oos_start"]
    oos_end     = w["oos_end"]

    for gvkey in gvkeys_all:
        g_all = firm_daily.get(gvkey)
        if g_all is None or g_all.empty:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "ok": False,
                    "msg": "missing_firm_panel",
                }
            )
            continue

        # TRAINING SLICE
        g_train = g_all.loc[(g_all.index >= train_start) & (g_all.index <= train_end)].copy()
        if g_train.empty or len(g_train) < MIN_DAILY_ROWS:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "ok": False,
                    "msg": "too_few_daily_rows_train",
                    "n_daily_train": int(len(g_train)),
                }
            )
            continue

        g_train2 = g_train.reset_index().rename(columns={"index": "date"})
        g_train2 = (
            g_train2.dropna(subset=["date", "E", "L", "r"])
                    .query("E > 0 and L > 0")
                    .sort_values("date")
                    .reset_index(drop=True)
        )

        if len(g_train2) < MIN_DAILY_ROWS:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "ok": False,
                    "msg": "too_few_daily_rows_train_after_clean",
                    "n_daily_train": int(len(g_train2)),
                }
            )
            continue

        p_inv_seed = P0_INV
        p_em_seed = prev_p_em.get(gvkey, P0_EM)

        # ONE INVERSION PASS ON TRAIN WINDOW
        try:
            weekly_is = invert_assets_weekly_for_firm(
                g_train2,
                p_inv_seed,
                tau=T_INV,
                U=INVERT_U,
                n=INVERT_N,
                week_ending=WEEK_ENDING,
            )
        except Exception as e:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "ok": False,
                    "msg": f"fail_train_inversion:{type(e).__name__}:{str(e)[:180]}",
                    "n_daily_train": int(len(g_train2)),
                }
            )
            continue

        n_weekly_ret = int(weekly_is["dlogA"].notna().sum())
        if n_weekly_ret < MIN_WEEKLY_RETURNS:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "ok": False,
                    "msg": f"too_few_weekly_returns:{n_weekly_ret}",
                    "n_daily_train": int(len(g_train2)),
                    "n_weekly_train": int(len(weekly_is)),
                }
            )
            continue

        # ONE EM FIT ON TRAIN WINDOW
        try:
            p_hat_weekly, upd_one = fit_one_nig_window(
                weekly_is,
                p_em_seed,
                em_max_iter=EM_MAX_ITER,
                em_tol=EM_TOL,
                alpha_min=3.0,
            )
        except Exception as e:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "ok": False,
                    "msg": f"fail_train_em:{type(e).__name__}:{str(e)[:180]}",
                    "n_daily_train": int(len(g_train2)),
                    "n_weekly_train": int(len(weekly_is)),
                }
            )
            continue

        prev_p_em[gvkey] = p_hat_weekly

        # TRAIN-END PD
        last_is = weekly_is.sort_values("date").iloc[-1]
        pd_is = pd_terminal_nig_weekly(
            float(last_is["A_hat"]),
            float(last_is["L"]),
            p_hat_weekly,
            horizon_weeks=HORIZON_WEEKS,
        )

        roll_summary_rows.append(
            {
                "gvkey": gvkey,
                "train_start": train_start,
                "train_end": train_end,
                "oos_start": oos_start,
                "oos_end": oos_end,
                "ok": True,
                "msg": "ok",
                "alpha": float(p_hat_weekly.alpha),
                "beta": float(p_hat_weekly.beta),
                "delta": float(p_hat_weekly.delta),
                "mu": float(p_hat_weekly.mu),
                "emp_mean": float(upd_one.iloc[-1]["emp_mean"]),
                "mod_mean": float(upd_one.iloc[-1]["mod_mean"]),
                "emp_std": float(upd_one.iloc[-1]["emp_std"]),
                "mod_std": float(upd_one.iloc[-1]["mod_std"]),
                "alpha_required_from_theta": float(upd_one.iloc[-1]["alpha_required_from_theta"])
                    if np.isfinite(upd_one.iloc[-1]["alpha_required_from_theta"]) else np.nan,
                "alpha_hits_floor": bool(upd_one.iloc[-1]["alpha_hits_floor"]),
                "pd_date_is": pd.Timestamp(last_is["date"]),
                "A_used_is": float(last_is["A_hat"]),
                "L_used_is": float(last_is["L"]),
                "r_used_is": float(last_is["r"]),
                "PD_1y_is": float(pd_is),
                "n_daily_train": int(len(g_train2)),
                "n_weekly_train": int(len(weekly_is)),
                "n_weekly_returns_train": int(n_weekly_ret),
            }
        )

        weekly_is_store = weekly_is.copy()
        weekly_is_store["gvkey"] = gvkey
        weekly_is_store["train_end"] = train_end
        weekly_is_store["alpha"] = float(p_hat_weekly.alpha)
        weekly_is_store["beta"] = float(p_hat_weekly.beta)
        weekly_is_store["delta"] = float(p_hat_weekly.delta)
        weekly_is_store["mu"] = float(p_hat_weekly.mu)
        weekly_is_store["PD_1y_is"] = [
            pd_terminal_nig_weekly(a, l, p_hat_weekly, horizon_weeks=HORIZON_WEEKS)
            for a, l in zip(
                weekly_is_store["A_hat"].astype(float),
                weekly_is_store["L"].astype(float),
            )
        ]
        roll_weekly_is_rows.append(weekly_is_store)

        # OOS QUARTER
        g_oos = g_all.loc[(g_all.index >= oos_start) & (g_all.index <= oos_end)].copy()
        if g_oos.empty:
            continue

        g_oos2 = g_oos.reset_index().rename(columns={"index": "date"})
        g_oos2 = (
            g_oos2.dropna(subset=["date", "E", "L", "r"])
                  .query("E > 0 and L > 0")
                  .sort_values("date")
                  .reset_index(drop=True)
        )
        if g_oos2.empty:
            continue

        try:
            weekly_oos = invert_assets_weekly_for_firm(
                g_oos2,
                P0_INV,
                tau=T_INV,
                U=INVERT_U,
                n=INVERT_N,
                week_ending=WEEK_ENDING,
            )
        except Exception as e:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "ok": False,
                    "msg": f"fail_oos_inversion:{type(e).__name__}:{str(e)[:180]}",
                }
            )
            continue

        if weekly_oos.empty:
            continue

        upd_oos = one_row_param_update(train_end, p_hat_weekly)

        try:
            pd_oos = pd_weekly_one_firm(
                weekly_oos,
                gvkey=str(gvkey),
                param_updates=upd_oos,
                p0=p_hat_weekly,
                horizon_weeks=HORIZON_WEEKS,
            )
        except Exception as e:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "ok": False,
                    "msg": f"fail_oos_pd:{type(e).__name__}:{str(e)[:180]}",
                }
            )
            continue

        oos_df_this = weekly_oos.merge(
            pd_oos[["date", "PD_1y"]],
            on="date",
            how="left",
        ).rename(columns={"PD_1y": "PD_1y_oos"})

        oos_df_this["gvkey"] = gvkey
        oos_df_this["train_end"] = train_end
        oos_df_this["alpha"] = float(p_hat_weekly.alpha)
        oos_df_this["beta"] = float(p_hat_weekly.beta)
        oos_df_this["delta"] = float(p_hat_weekly.delta)
        oos_df_this["mu"] = float(p_hat_weekly.mu)

        roll_weekly_oos_rows.append(oos_df_this)

print("Total runtime (sec):", round(time.time() - t0_all, 2))

Rolling NIG windows: 100%|██████████| 36/36 [42:22<00:00, 70.61s/it] 

Total runtime (sec): 2542.13


In [ ]:
print("len(roll_summary_rows):", len(roll_summary_rows))
print("len(roll_weekly_is_rows):", len(roll_weekly_is_rows))
print("len(roll_weekly_oos_rows):", len(roll_weekly_oos_rows))

if len(roll_weekly_oos_rows):
    print("first OOS chunk columns:", roll_weekly_oos_rows[0].columns.tolist())
else:
    print("No OOS chunks were created.")

len(roll_summary_rows): 1512
len(roll_weekly_is_rows): 756
len(roll_weekly_oos_rows): 0
No OOS chunks were created.


In [ ]:
# Cell 7 — assemble outputs

roll_summary_nig_df = (
    pd.DataFrame(roll_summary_rows)
      .sort_values(["train_end", "gvkey", "ok"], ascending=[True, True, False])
      .reset_index(drop=True)
)

roll_weekly_nig_is_df = (
    pd.concat(roll_weekly_is_rows, ignore_index=True)
      .sort_values(["train_end", "gvkey", "date"])
      .reset_index(drop=True)
) if len(roll_weekly_is_rows) else pd.DataFrame()

if len(roll_weekly_oos_rows):
    roll_weekly_nig_oos_df = (
        pd.concat(roll_weekly_oos_rows, ignore_index=True)
          .sort_values(["train_end", "gvkey", "date"])
          .reset_index(drop=True)
    )
else:
    roll_weekly_nig_oos_df = pd.DataFrame(
        columns=["gvkey", "date", "alpha", "beta", "delta", "mu", "A_hat", "L", "PD_1y_oos", "train_end"]
    )

print("roll_summary_nig_df shape:", roll_summary_nig_df.shape)
print("roll_weekly_nig_is_df shape:", roll_weekly_nig_is_df.shape)
print("roll_weekly_nig_oos_df shape:", roll_weekly_nig_oos_df.shape)

display(roll_summary_nig_df.head())
display(roll_weekly_nig_is_df.head())
display(roll_weekly_nig_oos_df.head())

roll_summary_nig_df shape: (1512, 25)
roll_weekly_nig_is_df shape: (79296, 15)
roll_weekly_nig_oos_df shape: (0, 10)


,gvkey,train_start,train_end,oos_start,oos_end,ok,msg,alpha,beta,delta,...,alpha_required_from_theta,alpha_hits_floor,pd_date_is,A_used_is,L_used_is,r_used_is,PD_1y_is,n_daily_train,n_weekly_train,n_weekly_returns_train
0,100022,2014-01-01,2015-12-31,2016-01-01,2016-03-31,True,ok,3.622577,0.005430,0.000761,...,3.622577,True,2015-12-31,1.766054e+11,1.173660e+11,-0.003968,0.003364,522.0,105.0,104.0
1,100022,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100080,2014-01-01,2015-12-31,2016-01-01,2016-03-31,True,ok,3.618602,0.001455,0.003083,...,3.618602,True,2015-12-31,1.459754e+11,5.001600e+10,-0.003968,0.000501,522.0,105.0,104.0
3,100080,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100957,2014-01-01,2015-12-31,2016-01-01,2016-03-31,True,ok,3.607977,-0.009170,0.000456,...,3.607977,True,2015-12-31,9.971793e+10,5.798090e+10,-0.003968,0.001061,522.0,105.0,104.0


,date,E,L,r,A_hat,theta,logA,dlogA,gvkey,train_end,alpha,beta,delta,mu,PD_1y_is
0,2014-01-03,5.055556e+10,1.014480e+11,0.000991,1.519031e+11,2.574282,25.746508,NaN,100022,2015-12-31,3.622577,0.00543,0.000761,0.001448,0.003469
1,2014-01-10,5.005590e+10,1.014480e+11,0.000963,1.514063e+11,2.572173,25.743233,-0.003276,100022,2015-12-31,3.622577,0.00543,0.000761,0.001448,0.003540
2,2014-01-17,5.189801e+10,1.014480e+11,0.000996,1.532450e+11,2.574649,25.755304,0.012071,100022,2015-12-31,3.622577,0.00543,0.000761,0.001448,0.003284
3,2014-01-24,4.934555e+10,1.014480e+11,0.000794,1.507131e+11,2.559494,25.738644,-0.016660,100022,2015-12-31,3.622577,0.00543,0.000761,0.001448,0.003644
4,2014-01-31,4.865325e+10,1.014480e+11,0.000371,1.500636e+11,2.527818,25.734325,-0.004318,100022,2015-12-31,3.622577,0.00543,0.000761,0.001448,0.003744


,gvkey,date,alpha,beta,delta,mu,A_hat,L,PD_1y_oos,train_end


In [ ]:

# Cell 8 — build final_nig_df analogous to your Merton final_df
is_last = (
    roll_weekly_nig_is_df.sort_values(["gvkey", "train_end", "date"])
    .groupby(["gvkey", "train_end"], as_index=False)
    .tail(1)
    [["gvkey", "train_end", "date", "A_hat", "L", "PD_1y_is", "alpha", "beta", "delta", "mu"]]
    .rename(columns={"A_hat": "A_used", "L": "L_used", "PD_1y_is": "PD_1y"})
)

train_end_rows = pd.DataFrame(
    {
        "gvkey": is_last["gvkey"].astype(str),
        "date": pd.to_datetime(is_last["date"]),
        "alpha": is_last["alpha"].astype(float),
        "beta": is_last["beta"].astype(float),
        "delta": is_last["delta"].astype(float),
        "mu": is_last["mu"].astype(float),
        "A_used": is_last["A_used"].astype(float),
        "L_used": is_last["L_used"].astype(float),
        "PD_1y": is_last["PD_1y"].astype(float),
        "train_end_date": pd.to_datetime(is_last["train_end"]),
        "training_end": 1,
    }
)

oos_rows = pd.DataFrame(
    {
        "gvkey": roll_weekly_nig_oos_df["gvkey"].astype(str),
        "date": pd.to_datetime(roll_weekly_nig_oos_df["date"]),
        "alpha": roll_weekly_nig_oos_df["alpha"].astype(float),
        "beta": roll_weekly_nig_oos_df["beta"].astype(float),
        "delta": roll_weekly_nig_oos_df["delta"].astype(float),
        "mu": roll_weekly_nig_oos_df["mu"].astype(float),
        "A_used": roll_weekly_nig_oos_df["A_hat"].astype(float),
        "L_used": roll_weekly_nig_oos_df["L"].astype(float),
        "PD_1y": roll_weekly_nig_oos_df["PD_1y_oos"].astype(float),
        "train_end_date": pd.to_datetime(roll_weekly_nig_oos_df["train_end"]),
        "training_end": 0,
    }
)

final_nig_df = pd.concat([train_end_rows, oos_rows], ignore_index=True)

final_nig_df = (
    final_nig_df.sort_values(["gvkey", "date", "training_end"], ascending=[True, True, False])
                .drop_duplicates(subset=["gvkey", "date"], keep="first")
                .sort_values(["gvkey", "date"])
                .reset_index(drop=True)
)

final_nig_df["training_end"] = final_nig_df["training_end"].astype(int)

final_nig_df = final_nig_df[
    ["gvkey", "date", "alpha", "beta", "delta", "mu", "A_used", "L_used", "PD_1y", "train_end_date", "training_end"]
]

display(final_nig_df.head(20))
print(final_nig_df.shape)


,gvkey,date,alpha,beta,delta,mu,A_used,L_used,PD_1y,train_end_date,training_end
0,100022,2015-12-31,3.622577,0.005430,0.000761,1.447662e-03,1.766054e+11,1.173660e+11,0.003364,2015-12-31,1
1,100022,2016-03-31,3.589003,0.000200,0.001049,1.112929e-03,1.786210e+11,1.294100e+11,0.009677,2016-03-31,1
2,100022,2016-06-30,3.489059,0.000303,0.001041,5.733899e-04,1.698580e+11,1.294100e+11,0.017243,2016-06-30,1
3,100022,2016-09-30,3.455496,0.000253,0.001005,1.262647e-03,1.753827e+11,1.294100e+11,0.010408,2016-09-30,1
4,100022,2016-12-30,3.424522,0.000256,0.000991,1.594763e-03,1.839058e+11,1.294100e+11,0.006835,2016-12-31,1
5,100022,2017-03-31,3.322206,0.000322,0.000890,3.454992e-04,1.937022e+11,1.411720e+11,0.012108,2017-03-31,1
6,100022,2017-06-30,3.297637,0.000203,0.000814,7.212003e-04,1.910265e+11,1.411720e+11,0.010650,2017-06-30,1
7,100022,2017-09-29,3.288490,0.000217,0.000742,1.564317e-03,1.939052e+11,1.411720e+11,0.006555,2017-09-30,1
8,100022,2017-12-29,3.186297,0.000334,0.000658,9.273958e-04,1.944888e+11,1.411720e+11,0.007266,2017-12-31,1
9,100022,2018-03-30,3.120381,0.000690,0.000337,8.310890e-04,1.929734e+11,1.389350e+11,0.003575,2018-03-31,1


(756, 11)


In [ ]:

# Cell 9 — quick diagnostics

if not roll_summary_nig_df.empty:
    print("Unique train_end windows:", roll_summary_nig_df["train_end"].nunique())
    print("Share ok:", roll_summary_nig_df["ok"].mean())
    print(roll_summary_nig_df.loc[roll_summary_nig_df["ok"] == True, ["alpha", "beta", "delta", "mu"]].describe())

    g0 = roll_summary_nig_df.loc[roll_summary_nig_df["ok"] == True, "gvkey"].astype(str).unique()
    if len(g0):
        example = g0[0]
        display(
            roll_summary_nig_df[
                (roll_summary_nig_df["gvkey"] == example) & (roll_summary_nig_df["ok"] == True)
            ][
                ["gvkey", "train_end", "alpha", "beta", "delta", "mu", "PD_1y_is", "emp_std", "mod_std"]
            ].head(15)
        )

if not roll_weekly_nig_oos_df.empty and len(g0):
    display(
        roll_weekly_nig_oos_df[
            roll_weekly_nig_oos_df["gvkey"] == example
        ][
            ["gvkey", "train_end", "date", "alpha", "beta", "delta", "mu", "A_hat", "PD_1y_oos"]
        ].head(30)
    )
# Cell 10 — optional: inspect failures only

fail_view = roll_summary_nig_df.loc[roll_summary_nig_df["ok"] == False].copy()
display(fail_view.head(30))
print(fail_view["msg"].value_counts().head(20))

Unique train_end windows: 36
Share ok: 0.5
            alpha        beta       delta          mu
count  756.000000  756.000000  756.000000  756.000000
mean     3.874928    0.000002    0.002023    0.001045
std      1.185463    0.000711    0.001751    0.001892
min      3.028215   -0.009170    0.000198   -0.006619
25%      3.078145   -0.000139    0.000839   -0.000066
50%      3.237367   -0.000011    0.001478    0.000913
75%      4.336445    0.000145    0.002577    0.001974
max      6.145042    0.005430    0.009775    0.008374


,gvkey,train_end,alpha,beta,delta,mu,PD_1y_is,emp_std,mod_std
0,100022,2015-12-31,3.622577,0.005430,0.000761,1.447662e-03,0.003364,0.014490,0.014490
42,100022,2016-03-31,3.589003,0.000200,0.001049,1.112929e-03,0.009677,0.017099,0.017099
84,100022,2016-06-30,3.489059,0.000303,0.001041,5.733899e-04,0.017243,0.017276,0.017276
126,100022,2016-09-30,3.455496,0.000253,0.001005,1.262647e-03,0.010408,0.017052,0.017052
168,100022,2016-12-31,3.424522,0.000256,0.000991,1.594763e-03,0.006835,0.017013,0.017013
210,100022,2017-03-31,3.322206,0.000322,0.000890,3.454992e-04,0.012108,0.016364,0.016364
252,100022,2017-06-30,3.297637,0.000203,0.000814,7.212003e-04,0.010650,0.015714,0.015714
294,100022,2017-09-30,3.288490,0.000217,0.000742,1.564317e-03,0.006555,0.015022,0.015022
336,100022,2017-12-31,3.186297,0.000334,0.000658,9.273958e-04,0.007266,0.014372,0.014372
378,100022,2018-03-31,3.120381,0.000690,0.000337,8.310890e-04,0.003575,0.010394,0.010394


,gvkey,train_start,train_end,oos_start,oos_end,ok,msg,alpha,beta,delta,...,alpha_required_from_theta,alpha_hits_floor,pd_date_is,A_used_is,L_used_is,r_used_is,PD_1y_is,n_daily_train,n_weekly_train,n_weekly_returns_train
1,100022,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100080,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,100957,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,101202,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,101204,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,101336,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,101361,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,102296,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,14447,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,17436,2014-01-01,2015-12-31,2016-01-01,2016-03-31,False,fail_oos_inversion:TypeError:invert_assets_wee...,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


msg
fail_oos_inversion:TypeError:invert_assets_weekly_seeded() missing 1 required positional argument: 'p'    756
Name: count, dtype: int64


In [ ]:
# define a firm-window key
rs = roll_summary_nig_df.copy()
rs["window_key"] = rs["gvkey"].astype(str) + "|" + rs["train_end"].astype(str)

# 1) training success rows only
train_ok_rows = rs[(rs["ok"] == True) & (rs["msg"] == "ok")].copy()
n_train_success = train_ok_rows["window_key"].nunique()

# 2) windows that had any training-stage failure
train_fail_rows = rs[rs["msg"].fillna("").str.startswith("fail_train_")].copy()
n_train_fail = train_fail_rows["window_key"].nunique()

# 3) windows with OOS failure after training success
oos_fail_rows = rs[rs["msg"].fillna("").str.startswith("fail_oos_")].copy()
n_oos_fail = oos_fail_rows["window_key"].nunique()

# denominator = all intended firm-windows
n_total_windows = len(gvkeys_all) * len(windows)

print("Total intended firm-windows:", n_total_windows)
print("Training-success windows:", n_train_success, "=>", n_train_success / n_total_windows)
print("Training-failure windows:", n_train_fail, "=>", n_train_fail / n_total_windows)
print("OOS-failure windows:", n_oos_fail, "=>", n_oos_fail / n_total_windows)

# full success = had training success and no oos failure
train_success_keys = set(train_ok_rows["window_key"])
oos_fail_keys = set(oos_fail_rows["window_key"])
full_success_keys = train_success_keys - oos_fail_keys

print("Full train+OOS success windows:", len(full_success_keys), "=>", len(full_success_keys) / n_total_windows)

Total intended firm-windows: 756
Training-success windows: 756 => 1.0
Training-failure windows: 0 => 0.0
OOS-failure windows: 756 => 1.0
Full train+OOS success windows: 0 => 0.0


In [ ]:
# save final_df as CSV in the current working directory
output_path = Path.cwd() / ".." / "data" / "derived"
final_nig_df.to_csv(output_path / "NIG_weekly.csv", index=False)